In [ ]:
# OpenAI offers automatic prompt caching with no code changes — prompts >1,024 tokens are cached for 5-10 minutes, 
# with cache hits charged at 10% of the base input price.
# Anthropic provides explicit cache control via cache_control blocks in the standard messages.create() API, 
# with 5-minute or 1-hour cache durations and cache reads at 10% of base price.
# Both providers offer 90% savings on cached token reads, making caching highly cost-effective for repeated prompts.

import os
from getpass import getpass
from openai import OpenAI
import anthropic
import time
 
OPENAI_MODEL = "gpt-5-mini"
ANTHROPIC_MODEL = "claude-3-haiku-20240307"  # Change to a newer model if available on your API key

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")

openai_client = OpenAI()
anthropic_client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

In [ ]:
# Part 1: OpenAI Automatic Prompt Caching (Responses API)
def check_openai_caching(system_prompt, user_prompt):
    """Make an OpenAI Responses API call and check cache usage.
    
    OpenAI automatically caches prompts >1024 tokens. The cached_tokens
    field in usage.input_tokens_details shows how many tokens were served
    from cache.
    """
    response = openai_client.responses.create(
        model=OPENAI_MODEL,
        instructions=system_prompt,
        input=user_prompt
    )
    
    usage = response.usage
    cached_tokens = 0
    if usage.input_tokens_details:
        cached_tokens = usage.input_tokens_details.cached_tokens or 0
    
    print(f"Input tokens:  {usage.input_tokens}")
    print(f"Cached tokens: {cached_tokens} ({cached_tokens / usage.input_tokens * 100:.0f}% cache hit)" if cached_tokens else f"Cached tokens: 0 (cache miss)")
    print(f"Output tokens: {usage.output_tokens}")
    print(f"Total tokens:  {usage.total_tokens}")
    
    return response

# First call - may or may not hit cache depending on prior requests
print("First call:")
response1 = check_openai_caching(long_system_prompt, "Explain quantum computing")

time.sleep(2)  # Small delay between calls

# Second call - should show caching (same prefix)
print("\nSecond call:")
response2 = check_openai_caching(long_system_prompt, "Explain neural networks")

First call:
Input tokens:  2066
Cached tokens: 0 (cache miss)
Output tokens: 1529
Total tokens:  3595

Second call:
Input tokens:  2066
Cached tokens: 1792 (87% cache hit)
Output tokens: 1417
Total tokens:  3483

In [ ]:
# Part 2: Anthropic Manual Prompt Caching . With Anthropic, we can explicitly control what gets cached using the cache_control parameter:
def create_anthropic_cached_message(system_content, user_message):
    """Create a message with cached system content in Anthropic.
    
    Prompt caching is now built into the standard messages.create() API.
    Use cache_control on content blocks to control what gets cached.
    """
    response = anthropic_client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=1024,
        system=[
            {
                "type": "text",
                "text": system_content,
                "cache_control": {"type": "ephemeral"}           # <<----------------------------------------
            }
        ],
        messages=[{"role": "user", "content": user_message}]
    )
    
    # Print usage statistics
    usage = response.usage
    print(usage)
    print(f"Cache creation tokens: {usage.cache_creation_input_tokens}")
    print(f"Cache read tokens: {usage.cache_read_input_tokens}")
    print(f"Regular input tokens: {usage.input_tokens}")
    print(f"Output tokens: {usage.output_tokens}")
    
    return response

In [ ]:
# Part 3: Cost Analysis
def calculate_cost_savings(cached_tokens, model):
    """Calculate cost savings from using cached tokens.
    
    Pricing (per million tokens):
    - Claude 3 Haiku: $0.25/MTok input, $0.03/MTok cache read (10% of base)
    - GPT-5 mini: $0.25/MTok input, $0.025/MTok cached input (10% of base)
    
    Update the prices dict below if using a different Anthropic model.
    """
    prices = {
        "claude-3-haiku-20240307": {
            "name": "Claude 3 Haiku",
            "base_input": 0.25 / 1_000_000,      # $0.25/MTok
            "cache_read": 0.03 / 1_000_000,       # $0.03/MTok (10% of base)
        },
        "claude-sonnet-4-5-20250514": {
            "name": "Claude Sonnet 4.5",
            "base_input": 3.00 / 1_000_000,       # $3/MTok
            "cache_read": 0.30 / 1_000_000,       # $0.30/MTok (10% of base)
        },
        "gpt-5-mini": {
            "name": "GPT-5 mini",
            "base_input": 0.25 / 1_000_000,       # $0.25/MTok
            "cache_read": 0.025 / 1_000_000,      # $0.025/MTok (10% of base)
        }
    }
    
    model_prices = prices.get(model, prices["gpt-5-mini"])
    base_cost = cached_tokens * model_prices["base_input"]
    cached_cost = cached_tokens * model_prices["cache_read"]
    savings = base_cost - cached_cost
    
    print(f"Model: {model_prices['name']}")
    print(f"Base cost for {cached_tokens:,} tokens: ${base_cost:.6f}")
    print(f"Cost with caching: ${cached_cost:.6f}")
    print(f"Savings: ${savings:.6f} ({(savings/base_cost)*100:.1f}%)")
    
    return savings

# Example cost analysis for 10,000 cached tokens
print("Anthropic Claude Sonnet 4.5:")
anthropic_savings = calculate_cost_savings(10000, ANTHROPIC_MODEL)

print("\nOpenAI GPT-5 mini:")
openai_savings = calculate_cost_savings(10000, OPENAI_MODEL)

Anthropic Claude Sonnet 4.5:
Model: Claude 3 Haiku
Base cost for 10,000 tokens: $0.002500
Cost with caching: $0.000300
Savings: $0.002200 (88.0%)

OpenAI GPT-5 mini:
Model: GPT-5 mini
Base cost for 10,000 tokens: $0.002500
Cost with caching: $0.000250
Savings: $0.002250 (90.0%)
